In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display, HTML
from pathlib import Path
import os

# Caminho absoluto robusto para o banco de dados, mesmo no Voilà
project_root = Path.cwd().parents[1]
db_path = project_root / "data" / "sjur_recortes.db"

if not db_path.exists():
    raise FileNotFoundError(f"❌ Banco de dados não encontrado: {db_path}")


In [ ]:
def carregar_emails_processados():
    conn = sqlite3.connect(db_path)
    query = "SELECT message_id, data_processamento FROM emails_processados ORDER BY data_processamento DESC"
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df

def gerar_links_outlook(df):
    df['link_outlook'] = df['message_id'].apply(lambda mid: f'<a href="outlook:{mid}">{mid}</a>')
    df = df.rename(columns={"data_processamento": "Data de Processamento"})
    return df[["Data de Processamento", "link_outlook"]].rename(columns={"link_outlook": "Outlook Link"})

df_raw = carregar_emails_processados()
df_links = gerar_links_outlook(df_raw)

display(HTML("<h3>📧 Emails já processados</h3>"))
display(HTML(df_links.to_html(escape=False, index=False)))


In [ ]:

from ipywidgets import widgets

seletor_email = widgets.Dropdown(
    options=[(f"{row['id']} | {row['message_id'][:40]}...", row['id']) for _, row in df_emails.iterrows()],
    description='Email ID:',
    layout=widgets.Layout(width='100%')
)
display(seletor_email)


In [ ]:

def mostrar_detalhes(email_id):
    html = f"<hr><h4>📌 Detalhes para Email ID: {email_id}</h4>"
    display(HTML(html))

    # Tabela: publicacoes
    df_pub = carregar_dataframe(f"SELECT texto, tipo FROM publicacoes WHERE message_id = (SELECT message_id FROM emails_processados WHERE id = {email_id})")
    if not df_pub.empty:
        display(HTML("<b>🗞️ Publicações:</b>"))
        display(df_pub)
    else:
        display(HTML("<i>⚠️ Nenhuma publicação encontrada.</i>"))

    # Tabela: partes
    df_partes = carregar_dataframe(f"SELECT parte, papel FROM partes WHERE message_id = (SELECT message_id FROM emails_processados WHERE id = {email_id})")
    if not df_partes.empty:
        display(HTML("<b>👥 Partes:</b>"))
        display(df_partes)
    else:
        display(HTML("<i>⚠️ Nenhuma parte encontrada.</i>"))

    # Tabela: metadados
    df_meta = carregar_dataframe(f"SELECT chave, valor FROM metadados WHERE message_id = (SELECT message_id FROM emails_processados WHERE id = {email_id})")
    if not df_meta.empty:
        display(HTML("<b>🗂️ Metadados:</b>"))
        display(df_meta)
    else:
        display(HTML("<i>⚠️ Nenhum metadado encontrado.</i>"))


In [ ]:

botao = widgets.Button(description="🔍 Ver Detalhes", button_style='primary')

def on_click(b):
    mostrar_detalhes(seletor_email.value)

botao.on_click(on_click)
display(botao)
